# Deploy model as a webservice on Azure Container Instance

## Table of contents
1. [Prerequisites](#prerequisites)

2. [Initialize workspace](#workspace)

3. [Deploy Model to ACI](#deploymodel)

- a) [Create scoring file](#scoringfile)
- b) [Define Enviroment](#env)
- c) [Deployment configuration](#configfile)
- d  [Deploy Webservice](#webservice)
- e) [Test Webservice](#testservice)



### 1. Prerequisites <a id='prerequisites'></a>

In [3]:
import numpy as np 
import azureml.core

# display the core SDK version number
print("Azure ML SDK Version: ", azureml.core.VERSION)

Azure ML SDK Version:  1.61.0


### 2. Initialize workspace <a id='workspace'></a>

In [7]:
from azureml.core import Workspace
from azureml.core.model import Model

subscription_id = 'f2f70602-65d9-479b-8014-a1c92a06ef0e'
resource_group = 'Learn_MLOps'
workspace_name = 'MLOps'

ws = Workspace(subscription_id, resource_group, workspace_name)
print(ws.name, ws.resource_group, ws.location, sep='\n')

MLOps
Learn_MLOps
eastus2


### 3. Deploy model <a id='deploymodel'></a>

#### a) Create a scoring script <a id='scoringfile'></a>

In [ ]:
%%writefile score.py
import json
import numpy as np
import os
import glob
import joblib
import onnxruntime
from azureml.core.model import Model
from inference_schema.schema_decorators import input_schema, output_schema
from inference_schema.parameter_types.numpy_parameter_type import NumpyParameterType

def init():
    global model, scaler, input_name, label_name

    scaler_path = glob.glob(
        os.path.join(os.getenv('AZUREML_MODEL_DIR'), 'scaler', '**', '*.pkl'),
        recursive=True
    )[0]
    scaler = joblib.load(scaler_path)

    model_onnx = glob.glob(
        os.path.join(os.getenv('AZUREML_MODEL_DIR'), 'support-vector-classifier', '**', '*.onnx'),
        recursive=True
    )[0]
    model = onnxruntime.InferenceSession(model_onnx, None)
    input_name = model.get_inputs()[0].name
    label_name = model.get_outputs()[0].name


@input_schema('data', NumpyParameterType(np.array([[34.927778, 0.24, 7.3899, 83, 16.1000, 1016.51, 1]])))
@output_schema(NumpyParameterType(np.array([0])))
def run(data):
    try:
        data = scaler.transform(data.reshape(1, 7))
        result = model.run([label_name], {input_name: data.astype(np.float32)})[0]
    except Exception as e:
        return str(e)
    return result.tolist()


#### b) Define Environment <a id='env'></a>

In [ ]:
from azureml.core.environment import Environment
from azureml.core.conda_dependencies import CondaDependencies
from azureml.core.model import InferenceConfig

# Fresh Python 3.8 env — AzureML-Minimal uses py3.6 which is incompatible with scikit-learn>=1.0
env = Environment(name="weather-deploy-env")
conda_deps = CondaDependencies()
conda_deps.set_python_version("3.8")
env.python.conda_dependencies = conda_deps

In [ ]:
for pip_package in ["numpy", "onnxruntime", "joblib", "azureml-core", "azureml-defaults",
                    "scikit-learn>=1.0,<1.4", "inference-schema", "inference-schema[numpy-support]"]:
    env.python.conda_dependencies.add_pip_package(pip_package)

inference_config = InferenceConfig(entry_script='score.py',
                                    environment=env)

#### c) Deployment Configuration <a id='configfile'></a>

In [11]:
from azureml.core.webservice import AciWebservice

deployment_config = AciWebservice.deploy_configuration(cpu_cores = 1, memory_gb = 1, collect_model_data=True)

#### d) Deploy web service <a id='webservice'></a>

In [13]:
model1 = Model(ws, 'scaler')
model2 = Model(ws, 'support-vector-classifier')

service_name = 'weather-aci-prediction'

In [16]:
service = Model.deploy(ws, service_name, models=[model1, model2], inference_config=inference_config, deployment_config=deployment_config, overwrite=True)
service.wait_for_deployment(show_output = True)
print(service.state)

/tmp/ipykernel_23853/2522194965.py:1: FutureWarning: azureml.core.model:
To leverage new model deployment capabilities, AzureML recommends using CLI/SDK v2 to deploy models as online endpoint, 
please refer to respective documentations 
https://docs.microsoft.com/azure/machine-learning/how-to-deploy-managed-online-endpoints /
https://docs.microsoft.com/azure/machine-learning/how-to-attach-kubernetes-anywhere 
For more information on migration, see https://aka.ms/acimoemigration 
To disable CLI/SDK v1 deprecation warning set AZUREML_LOG_DEPRECATION_WARNING_ENABLED to 'False'
  service = Model.deploy(ws, service_name, models=[model1, model2], inference_config=inference_config, deployment_config=deployment_config, overwrite=True)


WebserviceException: WebserviceException:
	Message: There is no model with id scaler:2,support-vector-classifier:2 in MLOps
	InnerException None
	ErrorResponse 
{
    "error": {
        "message": "There is no model with id scaler:2,support-vector-classifier:2 in MLOps"
    }
}

In [17]:
print(service.get_logs())

/bin/bash: /azureml-envs/azureml_aae9930899053d29a1d162c049d1285e/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /azureml-envs/azureml_aae9930899053d29a1d162c049d1285e/lib/libtinfo.so.6: no version information available (required by /bin/bash)
/bin/bash: /azureml-envs/azureml_aae9930899053d29a1d162c049d1285e/lib/libtinfo.so.6: no version information available (required by /bin/bash)
2026-05-06T23:10:54,444658857+00:00 - rsyslog/run 
2026-05-06T23:10:54,450629566+00:00 - gunicorn/run 
bash: /azureml-envs/azureml_aae9930899053d29a1d162c049d1285e/lib/libtinfo.so.6: no version information available (required by bash)
2026-05-06T23:10:54,455145300+00:00 | gunicorn/run | 
2026-05-06T23:10:54,456216953+00:00 - nginx/run 
2026-05-06T23:10:54,457666919+00:00 | gunicorn/run | ###############################################
2026-05-06T23:10:54,460219272+00:00 | gunicorn/run | AzureML Container Runtime Information
2026-05-06T23:10:54,464189643+00:00 | gunico

In [18]:
service.update(enable_app_insights=True)

#### e) Test web service <a id='testservice'></a>

In [19]:
print(service.scoring_uri)

http://ca4a12d0-8283-4a64-815f-31b29745cc69.eastus2.azurecontainer.io/score


In [20]:
print(service.swagger_uri)

http://ca4a12d0-8283-4a64-815f-31b29745cc69.eastus2.azurecontainer.io/swagger.json


In [21]:
service.state

'Healthy'

In [22]:
import json


input_payload = json.dumps({
    'data': [[34.927778, 0.24, 7.3899, 83, 16.1000, 1016.51, 1]],
    'method': 'predict'  # If you have a classification model, you can get probabilities by changing this to 'predict_proba'.
})

output = service.run(input_payload)

print(output)

[1]


In [ ]:
# service.delete()